# 1 · Data pipeline

Builds the full training dataset used everywhere downstream: 498 point-to-point
reaching trajectories for a 2-link, 2-arm model, sampled at 100 timesteps
each, plus each trajectory's target and which arm is reaching.

This notebook is **standalone** — every function it uses is defined here,
no external project modules required (only numpy/scipy/torch/matplotlib).

The lab only ever hand-measured **21** time/distance points describing how
a reaching movement unfolds in time (a bell-shaped velocity profile, peak
speed at ~42% of the movement — not 50%). Everything else is generated
from those 21 points:

1. **Fit a timing curve** through the 21 points and resample it at 100
   points — this is what gives the Hopf oscillators (8–13 Hz mu band, see
   notebook 02) enough timesteps per trajectory to actually complete
   several oscillation cycles.
2. **2-link-arm kinematics**: for every reachable target on a spatial
   grid, sweep the nearer arm along a circular arc from home to target,
   parametrised by the timing curve from step 1. The other arm stays
   parked at home.
3. **Derive labels**: which arm was active, and its target position.
4. Wrap it all in a PyTorch `Dataset` + `DataLoader`.

In [ ]:
from pathlib import Path

# This notebook lives in notebooks/, one level below the project root.
# All data/output paths are resolved relative to PROJECT_ROOT so the
# notebook works regardless of where Jupyter's cwd ends up.
PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())

In [ ]:
import pickle
from math import atan, cos, sin, pi, acos

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.spatial import distance as spdist
from torch.utils.data import Dataset, DataLoader

TIME_VS_DIST_FILE = PROJECT_ROOT / "data" / "raw" / "time_vs_dist.txt"   # 21-point seed curve
TRAJ_FILE = PROJECT_ROOT / "data" / "raw" / "splined_trajectories_100.txt"
TARGET_FILE = PROJECT_ROOT / "data" / "processed" / "targets_100.pkl"
ARM_FILE = PROJECT_ROOT / "data" / "processed" / "active_arms_100.pkl"

L_S, L_E = 0.3, 0.3   # arm segment lengths (shoulder / elbow)

## Step 1 — fit and resample the timing curve

Human reaching movements have an **asymmetric bell-curve velocity
profile** — peak velocity happens at roughly 42% of the movement
duration, not the midpoint. A plain sigmoid can't capture that asymmetry;
we fit a *Richards / generalised-logistic* curve (an extra shape
parameter skews the curve) through the 21 hand-measured points, then
resample it at 100 evenly spaced points.

In [ ]:
def gen_logistic(t_pct, k, t0, nu):
    x = t_pct / 100.0
    raw = (1.0 + np.exp(-k * (x - t0))) ** (-1.0 / nu)
    lo = (1.0 + np.exp(-k * (0.0 - t0))) ** (-1.0 / nu)
    hi = (1.0 + np.exp(-k * (1.0 - t0))) ** (-1.0 / nu)
    return 100.0 * (raw - lo) / (hi - lo)


def fit_timing_curve(n_points=100, seed_file=TIME_VS_DIST_FILE):
    with open(seed_file, "rb") as f:
        orig = np.array(pickle.load(f))   # (21, 2)

    t_orig, d_orig = orig[:, 0], orig[:, 1]

    (k_fit, t0_fit, nu_fit), _ = curve_fit(
        gen_logistic, t_orig, d_orig,
        p0=[10.0, 0.45, 1.0],
        bounds=([1, 0.1, 0.1], [30, 0.9, 5.0])
    )

    t_new = np.linspace(0, 100, n_points)
    d_new = gen_logistic(t_new, k_fit, t0_fit, nu_fit)
    d_new[0], d_new[-1] = 0.0, 100.0

    return t_orig, d_orig, t_new, d_new, (k_fit, t0_fit, nu_fit)


t_orig, d_orig, t_new, d_new, (k_fit, t0_fit, nu_fit) = fit_timing_curve(n_points=100)
print(f"Fitted parameters:  k={k_fit:.4f}  t0={t0_fit:.4f}  nu={nu_fit:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_orig, d_orig, "o", color="crimson", ms=8, label="Original 21 points")
axes[0].plot(t_new, d_new, "-", color="steelblue", lw=2, label="Fitted + resampled (100 pts)")
axes[0].set_xlabel("Time (%)"); axes[0].set_ylabel("Distance covered (%)")
axes[0].set_title("Timing curve: 21 -> 100 points")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

vel = np.gradient(d_new, t_new)
axes[1].plot(t_new, vel / vel.max(), color="darkorange", lw=2)
axes[1].axvline(t_new[np.argmax(vel)], color="gray", ls="--", lw=1)
axes[1].set_xlabel("Time (%)"); axes[1].set_ylabel("Normalised velocity")
axes[1].set_title(f"Velocity profile — peak at ~{t_new[np.argmax(vel)]:.0f}% of movement")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2 — generate trajectories (2-link-arm kinematics)

Two arms (`left`, `right`), each a 2-link manipulator with segment
lengths `L_s`/`L_e` = 0.3. For every target on a 29×29 grid, whichever
arm is closer sweeps along a **circular arc** from home to target;
`d_new` (the 100 distance-percentages from Step 1) parametrises how far
along the arc the arm is at each timestep — this is where the timing
profile gets baked into the spatial trajectory. Each trajectory is
`(100, 4)`: `[x_left, y_left, x_right, y_right]`.

In [ ]:
def get_angle(p1, p2):
    ang = abs(atan((p2[1] - p1[1]) / (p2[0] - p1[0])))
    if p1[0] < p2[0] and p1[1] < p2[1]:
        return ang
    elif p1[0] > p2[0] and p1[1] < p2[1]:
        return pi - ang
    elif p1[0] > p2[0] and p1[1] > p2[1]:
        return pi + ang
    else:
        return 2 * pi - ang


def fwd_left(mn):
    ts = ((mn[2] - mn[3]) * pi / 2) + pi / 2
    te = ((mn[0] - mn[1]) * pi / 2) + pi / 2
    return ((L_S + L_E * cos(te)) * cos(ts) + L_E * sin(te) * sin(ts) - 0.15,
            (L_S + L_E * cos(te)) * sin(ts) - L_E * sin(te) * cos(ts))


def fwd_right(mn):
    ts = ((mn[2] - mn[3]) * pi / 2) + pi / 2
    te = ((mn[0] - mn[1]) * pi / 2) + pi / 2
    return ((-L_S - L_E * cos(te)) * cos(ts) - L_E * sin(te) * sin(ts) + 0.15,
            -(-L_S - L_E * cos(te)) * sin(ts) - L_E * sin(te) * cos(ts))


def find_centre(x1, y1, x2, y2, r):
    xd, yd = x1 - x2, y1 - y2
    s = x1**2 - x2**2 + y1**2 - y2**2
    cy = (s / (2 * yd)) - y1
    a = 1 + (xd / yd) ** 2
    b = -2 * ((x1 * yd + cy * xd) / yd)
    c = x1**2 + cy**2 - r**2
    det = np.sqrt(b**2 - 4 * a * c)
    cx1 = (-b + det) / (2 * a)
    cx2 = (-b - det) / (2 * a)
    return cx1, (s / (2 * yd)) - (xd / yd) * cx1, \
        cx2, (s / (2 * yd)) - (xd / yd) * cx2


def circle_params(p1, p2, tang):
    d = spdist.euclidean(p1, p2)
    r = (d**2 / (4 * tang) + tang) / 2
    cx1, cy1, cx2, cy2 = find_centre(p1[0], p1[1], p2[0], p2[1], r)
    return (cx1, cy1, cx2, cy2), r, 2 * acos((r - tang) / r)


def generate_trajectories(d_new, curvature=0.07, grid_n=29):
    arm_left, arm_right = [-0.15, 0], [0.15, 0]
    home_left = fwd_left([0.8, 0.2, 0.8, 0.2])
    home_right = fwd_right([0.8, 0.2, 0.8, 0.2])

    x_grid = list(np.linspace(-0.7, 0.7, grid_n))
    y_grid = list(np.linspace(0.31, 0.60, grid_n))

    trajectories = []
    for tgt in [[x, y] for x in x_grid for y in y_grid]:
        if tgt[0] < 0:   # LEFT arm reaches
            if spdist.euclidean(arm_left, tgt) > 0.57:
                continue
            dist = spdist.euclidean(home_left, tgt)
            c, r, theta = circle_params(home_left, tgt, curvature * dist)
            cx, cy = c[2], c[3]
            th_h = get_angle((cx, cy), home_left)
            ee = []
            for dp in d_new:
                tc = dp / 100 * theta
                if tgt[1] < home_left[1]:
                    ee.append([r * cos(th_h - tc) + cx, r * sin(th_h - tc) + cy,
                               home_right[0], home_right[1]])
                else:
                    ee.append([r * cos(th_h + tc) + cx, r * sin(th_h + tc) + cy,
                               home_right[0], home_right[1]])
            trajectories.append(np.array(ee, dtype=np.float32))

        else:             # RIGHT arm reaches
            if spdist.euclidean(arm_right, tgt) > 0.57:
                continue
            dist = spdist.euclidean(home_right, tgt)
            c, r, theta = circle_params(home_right, tgt, curvature * dist)
            cx, cy = c[0], c[1]
            th_h = get_angle((cx, cy), home_right)
            ee = []
            for dp in d_new:
                tc = dp / 100 * theta
                if tgt[1] < home_right[1]:
                    ee.append([home_left[0], home_left[1],
                               r * cos(th_h + tc) + cx, r * sin(th_h + tc) + cy])
                else:
                    ee.append([home_left[0], home_left[1],
                               r * cos(th_h - tc) + cx, r * sin(th_h - tc) + cy])
            trajectories.append(np.array(ee, dtype=np.float32))

    return trajectories


trajectories = generate_trajectories(d_new)

print(f"Total trajectories : {len(trajectories)}")
print(f"Each trajectory    : {trajectories[0].shape}  (timesteps x 4 coords)")

fig, ax = plt.subplots(figsize=(6, 6))
sample_idx = [5, 60, 130, 230, 330, 430]
cmap = plt.cm.tab10
for i, idx in enumerate(sample_idx):
    traj = trajectories[idx]
    lT = np.linalg.norm(traj[-1, :2] - traj[0, :2])
    rT = np.linalg.norm(traj[-1, 2:] - traj[0, 2:])
    xs, ys = (traj[:, 0], traj[:, 1]) if lT > rT else (traj[:, 2], traj[:, 3])
    col = cmap(i)
    ax.plot(xs, ys, "-", color=col, lw=1.8, alpha=0.9)
    ax.scatter(xs[0], ys[0], color=col, s=70, marker="o", zorder=6)
    ax.scatter(xs[-1], ys[-1], color=col, s=90, marker="*", zorder=6)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_title("Sample reaching trajectories (o=start, *=target)")
ax.grid(True, alpha=0.3); ax.set_aspect("equal")
plt.show()

## Step 3 — derive targets and active-arm labels

For each trajectory: compare how far each arm travelled (start -> end
distance). Whichever arm travelled further is the "active" one; its final
position is the target (trajectories are generated to end exactly at the
target, so this is exact).

In [ ]:
def derive_targets_and_arms(trajectories):
    targets, active_arms = [], []

    for traj in trajectories:
        traj = np.array(traj, dtype=np.float32)
        left_pos, right_pos = traj[:, :2], traj[:, 2:]

        left_travel = np.linalg.norm(left_pos[-1] - left_pos[0])
        right_travel = np.linalg.norm(right_pos[-1] - right_pos[0])

        if left_travel > right_travel:
            active_arms.append(0)
            targets.append(left_pos[-1].astype(np.float32))
        else:
            active_arms.append(1)
            targets.append(right_pos[-1].astype(np.float32))

    return targets, active_arms


targets, active_arms = derive_targets_and_arms(trajectories)

n_left = sum(1 for a in active_arms if a == 0)
n_right = sum(1 for a in active_arms if a == 1)
print(f"Active arm distribution: left={n_left}  right={n_right}  total={len(trajectories)}")

## Step 4 — persist to disk

Writes `data/raw/splined_trajectories_100.txt`,
`data/processed/targets_100.pkl`, `data/processed/active_arms_100.pkl` —
exactly what `TrajectoryDataset` loads below.

In [ ]:
TRAJ_FILE.parent.mkdir(parents=True, exist_ok=True)
TARGET_FILE.parent.mkdir(parents=True, exist_ok=True)

with open(TRAJ_FILE, "wb") as f:
    pickle.dump(trajectories, f)
with open(TARGET_FILE, "wb") as f:
    pickle.dump(targets, f)
with open(ARM_FILE, "wb") as f:
    pickle.dump(active_arms, f)

print("Saved:", TRAJ_FILE)
print("Saved:", TARGET_FILE)
print("Saved:", ARM_FILE)

## Step 5 — `TrajectoryDataset` + `DataLoader`

Thin `torch.utils.data.Dataset` wrapper around the three files just
written. `__getitem__` returns `(traj, target, active_arm)` — the 3-tuple
every training loop unpacks. (Notebook 02 redefines this same class so it
can train standalone too.)

In [ ]:
class TrajectoryDataset(Dataset):

    def __init__(self, root_dir):
        root_dir = Path(root_dir)

        with open(root_dir / "data" / "raw" / "splined_trajectories_100.txt", "rb") as f:
            self.trajectories = pickle.load(f)
        with open(root_dir / "data" / "processed" / "targets_100.pkl", "rb") as f:
            self.targets = pickle.load(f)
        with open(root_dir / "data" / "processed" / "active_arms_100.pkl", "rb") as f:
            self.active_arms = pickle.load(f)

        assert len(self.trajectories) == len(self.targets) == len(self.active_arms)
        print(f"Loaded {len(self.trajectories)} trajectories")

    def __len__(self):
        return len(self.trajectories)

    def __getitem__(self, idx):
        traj = np.asarray(self.trajectories[idx], dtype=np.float32)
        target = np.asarray(self.targets[idx], dtype=np.float32)
        active_arm = np.int64(self.active_arms[idx])
        return torch.from_numpy(traj), torch.from_numpy(target), torch.tensor(active_arm)


dataset = TrajectoryDataset(PROJECT_ROOT)

traj, target, active_arm = dataset[0]
print("traj shape:", traj.shape, "| target shape:", target.shape, "| active_arm:", active_arm.item())

loader = DataLoader(dataset, batch_size=16, shuffle=True)
traj_batch, target_batch, arm_batch = next(iter(loader))
print("batch traj:", traj_batch.shape, "| batch target:", target_batch.shape, "| batch arm:", arm_batch.shape)

**Next:** `02_model_and_training.ipynb` — the `HopfTrajectoryModel`
architecture, and training it on this dataset.